In [1]:
import pandas as pd
import numpy as np
rng = np.random.default_rng()
import matplotlib.pyplot as plt
import matplotlib as mpl
from skimage.measure import block_reduce

#### Functions

In [327]:
def multivariate_lognormal_cascade(n, sigma1=1, sigma2=1, corr=0):
    mu1 = -1/2 * sigma1**2
    mu2 = -1/2 * sigma2**2

    PQ = np.exp(rng.multivariate_normal(np.array([mu1,mu2]),  np.array([[sigma1**2,corr*sigma1*sigma2], [corr*sigma1*sigma2, sigma2**2]]), 4))

    P = PQ[:,0].reshape(2,2)
    Q = PQ[:,1].reshape(2,2)
    for i in range(n):
        PQ = np.exp(rng.multivariate_normal(np.array([mu1,mu2]),np.array([[sigma1**2,corr*sigma1*sigma2], [corr*sigma1*sigma2, sigma2**2]]), P.shape[0]**2 * 4))
        P = np.kron(P, np.ones((2,2)))
        P = P * PQ[:,0].reshape(P.shape)
        Q = np.kron(Q, np.ones((2,2)))
        Q = Q * PQ[:,1].reshape(P.shape)

    P = P / np.sum(P)
    Q = Q / np.sum(Q)
    return(np.stack([P,Q], axis=-1))

def multifractal_estimation(pop):  
    mu1, mu2, s1, s2, corrs = [], [], [], [], []
    sizes = 2**np.arange(np.log2(lnc.shape[0])).astype(np.int32)
    for i in sizes:
        temp = block_reduce(pop, block_size = (i,i,1), func = np.sum)
        s1.append(np.var(np.log(temp[:,:,0])))
        mu1.append(np.mean(np.log(temp[:,:,0])))
        s2.append(np.var(np.log(temp[:,:,1])))
        mu2.append(np.mean(np.log(temp[:,:,1])))
        corrs.append(np.corrcoef(np.log(temp[:,:,0]).flatten(), np.log(temp[:,:,1]).flatten())[1,0])
    return(pd.DataFrame({'sizes': sizes, 's1s': s1, 'mu1s' : mu1 , 's2s' : s2, 'mu2s' : mu2, 'corrs': corrs}))

#### Analyzes

In [276]:
sizes = 2**np.arange(8) 
sigma1, sigma2, corr = 0.2, 0.4, 0.7

In [347]:
lnc = multivariate_lognormal_cascade(6, sigma1=sigma1, sigma2=sigma2, corr=corr)
multifractal_estimation(lnc)

,sizes,s1s,mu1s,s2s,mu2s,corrs
0,1,0.276120,-9.844532,1.124223,-10.276107,0.744540
1,2,0.246421,-8.443402,1.009085,-8.831337,0.748891
2,4,0.208798,-7.038200,0.858565,-7.370591,0.758898
3,8,0.168797,-5.631863,0.711638,-5.907416,0.773301
4,16,0.132574,-4.227042,0.559673,-4.446650,0.778001
5,32,0.090661,-2.819225,0.372193,-2.976821,0.832802
6,64,0.053173,-1.412664,0.266945,-1.523767,0.856923


In [24]:
sigma1**2, - np.sum((mu2 - np.mean(s1)) * (np.log2(sizes) - np.log2(sizes).mean())) / np.sum((np.log2(sizes) - np.log2(sizes).mean())**2)

(0.25, 0.3324885924791276)